# Task 138: namaster_pseudo — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: CMB power spectrum estimation using NaMaster pseudo-Cl

**Paper**: A unified pseudo-Cℓ framework
**Repository**: https://github.com/LSSTDESC/NaMaster

### Physical Background

Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.

### Forward Model

```
y = H * x + n  (PSF convolution)
```

### Inverse Problem

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

### Performance Metrics
- **PSNR**: 29.13 dB
- **SSIM**: N/A
- **Evaluation**: 1D power spectrum deconvolution (Cℓ recovery from masked sky)

### Mathematical Formulation

A measured spectrum is modeled as a superposition of spectral profiles:

$$S(\nu) = \sum_{k=1}^{K} A_k \, P_k(\nu; \theta_k) + B(\nu) + \epsilon(\nu)$$

where $P_k$ is a peak profile (Gaussian/Lorentzian/Voigt), $B(\nu)$ is the baseline, and $\epsilon$ is noise.

**Voigt profile**: $V(\nu) = \int_{-\infty}^{\infty} G(\nu'; \sigma) \, L(\nu - \nu'; \gamma) \, d\nu'$

**Nonlinear least-squares fit**:
$$\hat{\theta} = \arg\min_\theta \sum_i \left(\frac{S_i^{\text{obs}} - S_i^{\text{model}}(\theta)}{\sigma_i}\right)^2$$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
"""
namaster_pseudo - Pseudo-Cl Power Spectrum Deconvolution
========================================================
Task: Estimate true angular power spectrum from masked sky observations
Repo: https://github.com/LSSTDESC/NaMaster

Forward problem:
    Given a true power spectrum Cl, generate a full-sky CMB map via
    healpy.synfast, then apply a partial-sky mask. The observed pseudo-Cl
    from the masked map is biased by mode-coupling.

Inverse problem:
    Use NaMaster (pymaster) to compute the mode-coupling matrix from the
    mask and decouple the pseudo-Cl to recover the true Cl.

Usage:
    /data/yjh/namaster_pseudo_env/bin/python namaster_pseudo_code.py
"""

import numpy as np
import matplotlib

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`compute_pseudo_cl`

In [ ]:
def compute_pseudo_cl(full_map, mask, nside, lmax):
    """
    Naive pseudo-Cl: anafast on masked map, divided by f_sky.
    This is the *biased* estimate (forward observation).
    """
    masked_map = full_map * mask
    cl_pseudo = hp.anafast(masked_map, lmax=lmax)
    f_sky = np.mean(mask ** 2)
    cl_pseudo_corrected = cl_pseudo / f_sky
    return cl_pseudo_corrected

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
y = H * x + n  (PSF convolution)
```

Functions: `generate_data`

In [ ]:
def generate_data(nside=64, lmax=None):
    """
    Generate simulated CMB map with known Cl and a galactic mask.

    Returns
    -------
    cl_true : array, shape (lmax+1,)
        Input theoretical power spectrum.
    full_map : array, shape (npix,)
        Full-sky realization drawn from cl_true.
    mask_apo : array, shape (npix,)
        Apodized binary mask (galactic cut).
    nside : int
    lmax : int
    """
    if lmax is None:
        lmax = 2 * nside

    # ---- theoretical power spectrum (CMB-like) ----
    ell = np.arange(lmax + 1, dtype=float)
    cl_true = np.zeros(lmax + 1)
    cl_true[2:] = 1.0 / (ell[2:] * (ell[2:] + 1))
    cl_true *= 1e4  # scale for realistic amplitude

    # ---- full-sky realization ----
    np.random.seed(42)
    full_map = hp.synfast(cl_true, nside, lmax=lmax, verbose=False)

    # ---- galactic mask: cut |b| < 20° ----
    npix = hp.nside2npix(nside)
    mask = np.ones(npix)
    theta, _ = hp.pix2ang(nside, np.arange(npix))
    lat = np.pi / 2 - theta
    mask[np.abs(lat) < np.radians(20)] = 0

    # ---- apodize mask (C1 taper, 10° scale) ----
    mask_apo = nmt.mask_apodization(mask, 10.0, apotype='C1')

    return cl_true, full_map, mask_apo, nside, lmax

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
x_hat = argmin 1/2 ||H*x - y||^2 + lambda R(x)
```

Functions: `reconstruct_cl`

In [ ]:
def reconstruct_cl(full_map, mask, nside, lmax):
    """
    NaMaster deconvolution: compute the mode-coupling matrix from the
    mask and apply its inverse to obtain unbiased Cl estimates.

    Returns
    -------
    cl_decoupled : array – decoupled power spectrum in each bin
    ell_eff : array – effective multipole of each bin
    bins : NmtBin object
    """
    # NaMaster field: pass the *unmasked* map; NaMaster applies the mask
    f = nmt.NmtField(mask, [full_map])

    # Binning scheme – 4 multipoles per bin
    bin_size = 4
    b = nmt.NmtBin.from_nside_linear(nside, bin_size)

    # Workspace: compute the mode-coupling matrix
    w = nmt.NmtWorkspace()
    w.compute_coupling_matrix(f, f, b)

    # Decouple: pseudo-Cl -> true Cl
    cl_decoupled = nmt.compute_full_master(f, f, b)

    ell_eff = b.get_effective_ells()
    return cl_decoupled[0], ell_eff, b

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `compute_metrics`, `visualize`

In [ ]:
def compute_metrics(cl_true, cl_recon, ell_eff):
    """
    Evaluate reconstruction quality.

    Metrics
    -------
    PSNR (dB), Pearson correlation coefficient, RMSE, mean relative error.
    """
    # Interpolate true Cl at the effective ell values of each bin
    ell_all = np.arange(len(cl_true))
    cl_true_binned = np.interp(ell_eff, ell_all, cl_true)

    # Keep only ell >= 2 (monopole/dipole undefined)
    valid = ell_eff >= 2
    t = cl_true_binned[valid]
    r = cl_recon[valid]

    # PSNR
    data_range = np.max(t) - np.min(t)
    mse = np.mean((t - r) ** 2)
    psnr = 10 * np.log10(data_range ** 2 / mse) if mse > 0 else float('inf')

    # Pearson CC
    cc = float(np.corrcoef(t, r)[0, 1])

    # Relative error
    re = float(np.mean(np.abs(t - r) / (np.abs(t) + 1e-30)))

    # RMSE
    rmse = float(np.sqrt(mse))

    return {
        "psnr_dB": float(psnr),
        "correlation_coefficient": cc,
        "rmse": rmse,
        "mean_relative_error": re,
        "method": "NaMaster_pseudo_Cl_deconvolution",
    }

def visualize(cl_true, cl_pseudo, cl_recon, ell_eff, lmax, metrics, save_path):
    """Four-panel visualization of the deconvolution result."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ell_all = np.arange(len(cl_true))
    cl_true_at_ell = np.interp(ell_eff, ell_all, cl_true)

    # helper: D_ell = ell*(ell+1)*Cl / 2pi
    def D(ell_arr, cl_arr):
        return ell_arr * (ell_arr + 1) * cl_arr / (2 * np.pi)

    # ---- (a) True power spectrum ----
    ax = axes[0, 0]
    ax.plot(ell_all[2:], D(ell_all[2:], cl_true[2:]), 'b-', lw=1.5)
    ax.set_xlabel('ℓ')
    ax.set_ylabel('ℓ(ℓ+1)Cℓ / 2π')
    ax.set_title('(a) True Power Spectrum')
    ax.set_xlim([2, lmax])

    # ---- (b) Pseudo-Cl (naive) vs True ----
    ax = axes[0, 1]
    ax.plot(ell_all[2:], D(ell_all[2:], cl_pseudo[2:]), 'r-', alpha=0.7, lw=1, label='Pseudo-Cℓ')
    ax.plot(ell_all[2:], D(ell_all[2:], cl_true[2:]), 'b--', alpha=0.5, lw=1, label='True')
    ax.set_xlabel('ℓ')
    ax.set_ylabel('ℓ(ℓ+1)Cℓ / 2π')
    ax.set_title('(b) Pseudo-Cℓ (biased) vs True')
    ax.legend()
    ax.set_xlim([2, lmax])

    # ---- (c) Decoupled Cl (NaMaster) vs True ----
    ax = axes[1, 0]
    ax.plot(ell_eff, D(ell_eff, cl_recon), 'go-', ms=3, lw=1.5, label='NaMaster')
    ax.plot(ell_eff, D(ell_eff, cl_true_at_ell), 'b--', lw=1, label='True')
    ax.set_xlabel('ℓ')
    ax.set_ylabel('ℓ(ℓ+1)Cℓ / 2π')
    ax.set_title('(c) Decoupled Cℓ (NaMaster) vs True')
    ax.legend()
    ax.set_xlim([2, lmax])

    # ---- (d) Relative error per bin ----
    ax = axes[1, 1]
    valid = ell_eff >= 2
    rel_err = np.abs(cl_recon[valid] - cl_true_at_ell[valid]) / (np.abs(cl_true_at_ell[valid]) + 1e-30)
    ax.semilogy(ell_eff[valid], rel_err, 'k.-', ms=3)
    ax.axhline(y=0.1, color='r', ls='--', alpha=0.5, label='10% error')
    ax.set_xlabel('ℓ')
    ax.set_ylabel('|Cℓ_recon − Cℓ_true| / Cℓ_true')
    ax.set_title('(d) Relative Error per ℓ-bin')
    ax.legend()
    ax.set_xlim([2, lmax])

    fig.suptitle(
        f"NaMaster Pseudo-Cℓ Deconvolution  |  "
        f"PSNR={metrics['psnr_dB']:.2f} dB  |  "
        f"CC={metrics['correlation_coefficient']:.4f}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"[VIS] Saved → {save_path}")

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
print("=" * 60)
print("  namaster_pseudo — Pseudo-Cℓ Deconvolution Pipeline")
print("=" * 60)

### Ground Truth Construction

Create or load the true underlying object/signal. Ground truth is needed for quantitative evaluation of the reconstruction.

In [ ]:
# (a) Generate data -------------------------------------------------
cl_true, full_map, mask, nside, lmax = generate_data(nside=64)
print(f"[DATA] nside={nside}, lmax={lmax}, npix={hp.nside2npix(nside)}")
print(f"[DATA] f_sky = {np.mean(mask**2):.3f}")

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (b) Naive pseudo-Cl (forward observation, biased) -----------------
cl_pseudo = compute_pseudo_cl(full_map, mask, nside, lmax)
print("[PSEUDO] Computed naive pseudo-Cℓ (f_sky-corrected)")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (c) NaMaster deconvolution (inverse) ------------------------------
cl_recon, ell_eff, bins = reconstruct_cl(full_map, mask, nside, lmax)
print(f"[RECON] Decoupled Cℓ: {len(ell_eff)} bins, "
      f"ℓ ∈ [{ell_eff[0]:.0f}, {ell_eff[-1]:.0f}]")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (d) Evaluate metrics ----------------------------------------------
metrics = compute_metrics(cl_true, cl_recon, ell_eff)
print(f"[EVAL] PSNR  = {metrics['psnr_dB']:.2f} dB")
print(f"[EVAL] CC    = {metrics['correlation_coefficient']:.6f}")
print(f"[EVAL] RMSE  = {metrics['rmse']:.6e}")
print(f"[EVAL] RE    = {metrics['mean_relative_error']:.4f}")

### Results Visualization and Analysis

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# (e) Save metrics --------------------------------------------------
metrics_path = os.path.join(RESULTS_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"[SAVE] Metrics → {metrics_path}")

### Inverse Problem — Reconstruction

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# (f) Visualize -----------------------------------------------------
vis_path = os.path.join(RESULTS_DIR, "reconstruction_result.png")
visualize(cl_true, cl_pseudo, cl_recon, ell_eff, lmax, metrics, vis_path)

### Forward Model — Generating Measurements

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# (g) Save arrays ---------------------------------------------------
np.save(os.path.join(RESULTS_DIR, "ground_truth.npy"), cl_true)
np.save(os.path.join(RESULTS_DIR, "recon_output.npy"), cl_recon)
np.save(os.path.join(RESULTS_DIR, "observed_data.npy"), cl_pseudo)
np.save(os.path.join(RESULTS_DIR, "ell_effective.npy"), ell_eff)
print("[SAVE] Arrays saved (ground_truth, recon_output, observed_data, ell_effective)")

### Processing Step

Intermediate processing step in the pipeline.

In [ ]:
print("=" * 60)
print("  DONE")
print("=" * 60)

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **namaster_pseudo**:

1. **Problem Setup**: Optical systems blur images via the Point Spread Function. Deconvolution inverts this to recover sharp detail.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=29.13 dB, SSIM=N/A)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: A unified pseudo-Cℓ framework
- Repository: https://github.com/LSSTDESC/NaMaster
